# Projet 13 — Contrôle qualité des stocks (version portfolio)

Version publique centrée sur les méthodes de contrôle et leur validation. Les sorties d’exécution ont été retirées pour ne pas exposer les données ERP et Web.

**Pour exécuter ce notebook :** placer le fichier `erp.xlsx` autorisé dans le même dossier. Les données sources ne sont pas incluses dans le dépôt public. Certaines cellules de diagnostic nécessitent l’exécution des cellules précédentes dans l’ordre.

**Important :** les résultats présentés dans la documentation correspondent à des exécutions réalisées sur les données d’origine, non fournies ici.


# Projet 13 – Amélioration du Projet 6 avec l'IA

## Étape 1 : Contrôle automatisé de la qualité des stocks

### Méthode 1 – Détection des incohérences avec Pandas

Objectif : identifier automatiquement les produits dont le
statut de stock ne correspond pas à la quantité disponible.

Cette méthode sera comparée à une seconde approche
pour évaluer sa fiabilité et sa reproductibilité.

In [ ]:

# P13 - Étape 1.1 : Détection automatique des anomalies de stock
# Méthode 1 - Pandas
# Suppression ciblée de l'avertissement OpenPyXL

import pandas as pd
import warnings

# 1. Masquer uniquement l'avertissement OpenPyXL identifié
warnings.filterwarnings(
    "ignore",
    message=r"Unknown extension is not supported and will be removed",
    category=UserWarning,
    module=r"openpyxl\..*"
)

# 2. Chargement des données ERP
erp_p13 = pd.read_excel("erp.xlsx")

# 3. Définition des règles de cohérence des stocks
anomalies_stock = erp_p13[
    (
        (erp_p13["stock_quantity"] > 0) &
        (erp_p13["stock_status"] != "instock")
    )
    |
    (
        (erp_p13["stock_quantity"] <= 0) &
        (erp_p13["stock_status"] != "outofstock")
    )
].copy()

# 4. Calcul des indicateurs de qualité
nb_produits = len(erp_p13)
nb_anomalies = len(anomalies_stock)

taux_anomalies = (
    nb_anomalies / nb_produits * 100
    if nb_produits > 0 else 0
)

# 5. Affichage des résultats
print("=== CONTROLE QUALITE DES STOCKS ===")
print(f"Nombre de produits : {nb_produits}")
print(f"Nombre d'anomalies : {nb_anomalies}")
print(f"Taux d'anomalies : {taux_anomalies:.2f} %")

# 6. Affichage des produits présentant une anomalie
display(
    anomalies_stock[
        ["product_id", "stock_quantity", "stock_status"]
    ]
)


In [ ]:
# P13 - Vérification du dossier de travail

import os

print("Dossier actuel :", os.getcwd())
print("\nFichiers disponibles :")
print(os.listdir())

## Étape 1.2 : Validation automatique des règles métier

### Méthode 2 – Comparaison avec un statut de stock attendu

**Objectif :** calculer automatiquement le statut attendu
pour chaque produit à partir de sa quantité disponible.

**Règles métier :**
- Stock supérieur à 0 : instock.
- Stock inférieur ou égal à 0 : outofstock.

Cette méthode sera comparée à la méthode 1 afin
d'évaluer la qualité des résultats, la lisibilité du code,
la reproductibilité et la facilité de maintenance.

In [ ]:
# P13 - Étape 1.2 : Méthode 2
# Validation automatique des statuts de stock

import numpy as np

# Créer une copie pour préserver les données originales
erp_validation = erp_p13.copy()

# Calculer le statut attendu
erp_validation["statut_attendu"] = np.where(
    erp_validation["stock_quantity"] > 0,
    "instock",
    "outofstock"
)

# Comparer le statut réel au statut attendu
erp_validation["anomalie_stock"] = (
    erp_validation["stock_status"] !=
    erp_validation["statut_attendu"]
)

# Extraire les anomalies
anomalies_methode2 = erp_validation[
    erp_validation["anomalie_stock"]
]

print("=== METHODE 2 : VALIDATION AUTOMATIQUE ===")
print("Produits analysés :", len(erp_validation))
print("Anomalies détectées :", len(anomalies_methode2))

display(
    anomalies_methode2[
        [
            "product_id",
            "stock_quantity",
            "stock_status",
            "statut_attendu"
        ]
    ]
)


## Étape 1.3 : Comparaison des méthodes de détection

**Objectif :** comparer les deux méthodes de contrôle des stocks.

- Méthode 1 : filtrage direct des incohérences avec Pandas.
- Méthode 2 : calcul du statut attendu avec NumPy et comparaison avec le statut réel.

**Critères :** résultats, lisibilité, traçabilité et maintenabilité.

Cette comparaison permettra de justifier le choix de la méthode retenue pour le livrable amélioré.


In [ ]:

# P13 - Étape 1.3 : Comparaison des méthodes 1 et 2

# 1. Récupérer les identifiants des produits détectés
ids_methode1 = set(anomalies_stock["product_id"])
ids_methode2 = set(anomalies_methode2["product_id"])

# 2. Comparer les résultats
resultats_identiques = ids_methode1 == ids_methode2

# 3. Construire le tableau comparatif
comparaison = pd.DataFrame({
    "Critere": [
        "Produits analyses",
        "Anomalies detectees",
        "Resultats identiques",
        "Approche",
        "Tracabilite du statut attendu"
    ],
    "Methode 1": [
        len(erp_p13),
        len(anomalies_stock),
        resultats_identiques,
        "Filtrage direct Pandas",
        "Non"
    ],
    "Methode 2": [
        len(erp_validation),
        len(anomalies_methode2),
        resultats_identiques,
        "Statut attendu avec NumPy",
        "Oui"
    ]
})

print("=== COMPARAISON DES DEUX METHODES ===")
display(comparaison)

print("\nProduits detectes uniquement par la methode 1 :",
      sorted(ids_methode1 - ids_methode2))

print("Produits detectes uniquement par la methode 2 :",
      sorted(ids_methode2 - ids_methode1))

print("\nConclusion :")
if resultats_identiques:
    print("Les deux methodes detectent les memes produits.")
else:
    print("Les methodes presentent des differences a analyser.")



## Étape 1.4 : Synthèse et justification du choix technique

### 1. Contexte métier

Dans le cadre de l'analyse des données de la boutique
Bottleneck, la fiabilité des informations de stock est
essentielle pour assurer une bonne gestion des produits
et limiter les erreurs de disponibilité.

### 2. Méthodes comparées

Deux méthodes de détection des incohérences ont été testées :

- Méthode 1 : filtrage direct avec Pandas.
- Méthode 2 : calcul du statut attendu avec NumPy,
  puis comparaison avec le statut enregistré.

### 3. Résultats obtenus

Les deux méthodes ont analysé 825 produits et détecté
2 anomalies, soit un taux d'incohérence de 0,24 %.

Les produits concernés sont :
- Produit 4039 : stock positif mais statut outofstock.
- Produit 4885 : stock nul mais statut instock.

Les deux méthodes ont identifié exactement les mêmes produits.

### 4. Décision retenue

La méthode 2 est retenue pour le livrable amélioré.

Elle permet :
- d'expliciter le statut attendu pour chaque produit ;
- de faciliter la compréhension des anomalies ;
- de conserver une trace exploitable du contrôle ;
- de simplifier les vérifications ultérieures.

### 5. Limites

Cette comparaison repose sur un seul jeu de données.

Les règles de contrôle supposent qu'un stock positif
correspond à un produit disponible à la vente.

Les valeurs manquantes, les statuts inconnus et les
éventuelles règles particulières de réservation ou
d'indisponibilité nécessitent des contrôles complémentaires.

### 6. Apport de l'intelligence artificielle

L'IA a été utilisée comme assistance pour proposer
les méthodes de contrôle, structurer leur comparaison
et faciliter la documentation.

Les résultats ont été vérifiés par exécution des
codes Python dans Jupyter Notebook.

La décision finale repose sur la comparaison des
résultats et sur les besoins de traçabilité métier.



## Étape 1.5 : Tests de robustesse de la méthode retenue

### Objectif

Évaluer la fiabilité de la méthode 2 face à des données
incomplètes ou incohérentes.

### Scénarios de test

1. Quantité de stock manquante.
2. Statut de stock manquant.
3. Statut inconnu.
4. Quantité de stock négative.

### Méthodologie

Les tests seront réalisés sur des données fictives
afin de préserver les données ERP originales.

Les résultats permettront d'identifier les limites
de la méthode et les améliorations nécessaires.


In [ ]:

# P13 - Étape 1.5 : Tests de robustesse
# Création de données fictives pour tester la méthode 2

import pandas as pd
import numpy as np

# 1. Création de scénarios de test
donnees_test = pd.DataFrame({
    "product_id": [9001, 9002, 9003, 9004, 9005],
    "stock_quantity": [10, 0, np.nan, 5, -3],
    "stock_status": [
        "instock",
        "outofstock",
        "instock",
        "unknown",
        "outofstock"
    ]
})

# 2. Application de la méthode 2 initiale
donnees_test["statut_attendu"] = np.where(
    donnees_test["stock_quantity"] > 0,
    "instock",
    "outofstock"
)

# 3. Détection des incohérences
donnees_test["anomalie_stock"] = (
    donnees_test["stock_status"] !=
    donnees_test["statut_attendu"]
)

# 4. Affichage des résultats
print("=== TESTS DE ROBUSTESSE - METHODE 2 ===")
display(donnees_test)



## Étape 1.6 : Amélioration de la méthode de contrôle

### Objectif

Renforcer la fiabilité du contrôle automatique des stocks
à partir des limites identifiées lors des tests de robustesse.

### Améliorations apportées

- Identifier les quantités manquantes.
- Détecter les quantités négatives.
- Identifier les statuts manquants.
- Détecter les statuts inconnus.
- Vérifier la cohérence entre la quantité et le statut.

### Principe

Les données invalides seront signalées séparément
des incohérences de statut.

Les données ERP originales seront préservées.


In [ ]:

# P13 - Étape 1.6 : Méthode 2 améliorée
# Contrôle robuste des stocks

import pandas as pd
import numpy as np

# 1. Préparer les données fictives
tests_ameliores = donnees_test[
    ["product_id", "stock_quantity", "stock_status"]
].copy()

# Ajouter un scénario avec un statut manquant
nouveau_test = pd.DataFrame({
    "product_id": [9006],
    "stock_quantity": [4],
    "stock_status": [np.nan]
})

tests_ameliores = pd.concat(
    [tests_ameliores, nouveau_test],
    ignore_index=True
)

# 2. Identifier les problèmes de qualité
tests_ameliores["quantite_manquante"] = (
    tests_ameliores["stock_quantity"].isna()
)

tests_ameliores["quantite_negative"] = (
    tests_ameliores["stock_quantity"] < 0
)

tests_ameliores["statut_manquant"] = (
    tests_ameliores["stock_status"].isna()
)

tests_ameliores["statut_inconnu"] = (
    tests_ameliores["stock_status"].notna()
    & ~tests_ameliores["stock_status"].isin(
        ["instock", "outofstock"]
    )
)

# 3. Calculer le statut attendu uniquement
# pour les quantités valides
tests_ameliores["statut_attendu"] = np.select(
    [
        tests_ameliores["stock_quantity"] > 0,
        tests_ameliores["stock_quantity"] == 0
    ],
    ["instock", "outofstock"],
    default=None
)

# 4. Détecter les incohérences de statut
# uniquement si les données sont valides
donnees_valides = (
    ~tests_ameliores["quantite_manquante"]
    & ~tests_ameliores["quantite_negative"]
    & ~tests_ameliores["statut_manquant"]
    & ~tests_ameliores["statut_inconnu"]
)

tests_ameliores["incoherence_statut"] = (
    donnees_valides
    & (
        tests_ameliores["stock_status"]
        != tests_ameliores["statut_attendu"]
    )
)

# 5. Déterminer les anomalies globales
colonnes_controle = [
    "quantite_manquante",
    "quantite_negative",
    "statut_manquant",
    "statut_inconnu",
    "incoherence_statut"
]

tests_ameliores["anomalie_globale"] = (
    tests_ameliores[colonnes_controle].any(axis=1)
)

# 6. Afficher les résultats
print("=== TEST DE LA METHODE AMELIOREE ===")
print("Produits testes :", len(tests_ameliores))
print(
    "Anomalies detectees :",
    tests_ameliores["anomalie_globale"].sum()
)

display(tests_ameliores)



## Étape 1.7 : Application aux données réelles de Bottleneck

### Objectif

Appliquer la méthode de contrôle améliorée aux données
ERP réelles de Bottleneck.

### Contrôles réalisés

- Quantités manquantes.
- Quantités négatives.
- Statuts manquants.
- Statuts inconnus.
- Incohérences entre quantité et statut.

### Résultat attendu

Identifier et quantifier les anomalies de qualité
sans modifier les données originales.

Comparer les résultats obtenus avec ceux de la
méthode initiale.


In [ ]:

# P13 - Étape 1.7 : Contrôle amélioré sur les données ERP

import pandas as pd
import numpy as np

# 1. Préserver les données originales
erp_controle = erp_p13.copy()

# 2. Vérifier la qualité des quantités
erp_controle["quantite_manquante"] = (
    erp_controle["stock_quantity"].isna()
)

erp_controle["quantite_negative"] = (
    erp_controle["stock_quantity"] < 0
)

# 3. Vérifier la qualité des statuts
erp_controle["statut_manquant"] = (
    erp_controle["stock_status"].isna()
)

erp_controle["statut_inconnu"] = (
    erp_controle["stock_status"].notna()
    & ~erp_controle["stock_status"].isin(
        ["instock", "outofstock"]
    )
)

# 4. Calculer le statut attendu
erp_controle["statut_attendu"] = np.select(
    [
        erp_controle["stock_quantity"] > 0,
        erp_controle["stock_quantity"] == 0
    ],
    ["instock", "outofstock"],
    default=None
)

# 5. Contrôler la cohérence des statuts
donnees_valides = (
    ~erp_controle["quantite_manquante"]
    & ~erp_controle["quantite_negative"]
    & ~erp_controle["statut_manquant"]
    & ~erp_controle["statut_inconnu"]
)

erp_controle["incoherence_statut"] = (
    donnees_valides
    & (
        erp_controle["stock_status"]
        != erp_controle["statut_attendu"]
    )
)

# 6. Identifier toutes les anomalies
colonnes_controle = [
    "quantite_manquante",
    "quantite_negative",
    "statut_manquant",
    "statut_inconnu",
    "incoherence_statut"
]

erp_controle["anomalie_globale"] = (
    erp_controle[colonnes_controle].any(axis=1)
)

# 7. Calculer les indicateurs
total_produits = len(erp_controle)
total_anomalies = int(
    erp_controle["anomalie_globale"].sum()
)

taux_anomalies = (
    total_anomalies / total_produits * 100
    if total_produits > 0 else 0
)

# 8. Afficher les résultats
print("=== CONTROLE QUALITE AMELIORE - BOTTLENECK ===")
print("Produits analyses :", total_produits)
print("Anomalies detectees :", total_anomalies)
print(f"Taux d'anomalies : {taux_anomalies:.2f} %")

print("\n=== DETAIL PAR TYPE D'ANOMALIE ===")
print(erp_controle[colonnes_controle].sum())

print("\n=== PRODUITS PRESENTANT UNE ANOMALIE ===")
display(
    erp_controle.loc[
        erp_controle["anomalie_globale"],
        [
            "product_id",
            "stock_quantity",
            "stock_status",
            "statut_attendu",
            *colonnes_controle
        ]
    ]
)



## Étape 1.8 : Bilan de l'amélioration du contrôle qualité

### 1. Résultats comparatifs

L'analyse initiale des 825 produits de Bottleneck
avait permis de détecter 2 anomalies de statut,
soit 0,24 % des produits.

Après amélioration de la méthode de contrôle,
4 produits présentant une anomalie ont été identifiés,
soit 0,48 % des produits.

La nouvelle méthode a donc détecté 2 anomalies
supplémentaires, correspondant à des quantités
de stock négatives.

### 2. Valeur ajoutée de l'amélioration

La méthode améliorée permet :

- de détecter plusieurs catégories d'anomalies ;
- de distinguer les problèmes de qualité des données
  des incohérences de statut ;
- de faciliter l'identification des produits à vérifier ;
- de préserver les données originales ;
- de rendre les contrôles plus explicites et reproductibles.

### 3. Apport de l'IA

L'intelligence artificielle a été utilisée pour
proposer des améliorations du code Python et
concevoir des scénarios de tests de robustesse.

Les propositions ont été vérifiées par exécution
dans Jupyter Notebook.

Les tests ont révélé des limites de la méthode initiale,
notamment concernant les valeurs manquantes et
les stocks négatifs.

### 4. Décision technique

La méthode améliorée est retenue pour le contrôle
qualité des stocks de Bottleneck.

Les anomalies détectées devront être examinées
avant toute correction des données sources.

### 5. Limites et prochaines étapes

Les contrôles reposent sur les règles métier définies
pour ce projet.

Les stocks négatifs nécessitent une vérification métier
afin d'en déterminer la cause.

La méthode devra être réévaluée si les règles de gestion
des stocks évoluent.

Des contrôles complémentaires sur les références
produits et le rapprochement ERP/Web pourront
être envisagés.



## Étape 1.9 : Journal des expérimentations avec l'IA

### Objectif

Assurer la traçabilité des propositions de l'intelligence
artificielle, des tests réalisés et des décisions retenues.

### Outil utilisé

- Assistant IA : ChatGPT
- Environnement : Jupyter Notebook
- Langage : Python
- Bibliothèques : Pandas et NumPy

### Expérimentation 1 : Détection des incohérences

**Proposition IA :**
Utiliser un filtrage conditionnel avec Pandas.

**Résultat :**
2 anomalies détectées sur 825 produits.

**Décision :**
Méthode conservée comme référence comparative.

### Expérimentation 2 : Calcul du statut attendu

**Proposition IA :**
Utiliser NumPy pour calculer automatiquement le
statut attendu à partir des quantités disponibles.

**Résultat :**
2 anomalies détectées, identiques à la méthode 1.

**Décision :**
Méthode retenue pour sa meilleure traçabilité métier.

### Expérimentation 3 : Tests de robustesse

**Proposition IA :**
Tester des quantités manquantes, des quantités négatives
et des statuts inconnus ou manquants.

**Résultat :**
La méthode initiale présente des limites face
à certaines données invalides.

**Décision :**
Renforcer les règles de validation.

### Expérimentation 4 : Méthode améliorée

**Proposition IA :**
Ajouter des contrôles spécifiques pour les quantités
et les statuts invalides.

**Résultat :**
4 anomalies détectées sur les 825 produits réels,
contre 2 avec la méthode initiale.

**Décision :**
Retenir la méthode améliorée pour le contrôle qualité.

### Limites de l'utilisation de l'IA

Les propositions générées par l'IA nécessitent
une vérification systématique.

Les résultats ont été contrôlés par exécution
des codes dans Jupyter Notebook.

Les décisions techniques ont été prises à partir
des résultats observés et des besoins métier.



## Étape 1.10 : Veille métier et technologique

### Objectif

Identifier et comparer des outils de contrôle
de la qualité des données afin de justifier
la solution retenue pour Bottleneck.

### Besoin métier

Fiabiliser les données de stock issues de l'ERP
pour faciliter la gestion des produits et limiter
les erreurs de disponibilité.

### Solutions à étudier

**Solution 1 : Pandas et NumPy**

Contrôles automatisés de la qualité des stocks
à partir de règles métier définies en Python.

**Solution 2 : Great Expectations**

Validation structurée des données à partir
de règles de qualité explicites.

### Critères de comparaison

- Qualité des contrôles.
- Facilité d'utilisation.
- Reproductibilité.
- Maintenance.
- Coût et temps de mise en œuvre.
- Sécurité et confidentialité des données.

### Résultat attendu

Produire un tableau de veille documenté,
comparer les solutions et justifier le choix
technique retenu.



## Étape 1.11 : Comparaison des solutions technologiques

### Objectif

Comparer Pandas/NumPy et Great Expectations
afin de sélectionner une solution adaptée
au contrôle qualité des stocks de Bottleneck.

### Tableau comparatif

| Critère | Pandas / NumPy | Great Expectations |
|---|---|---|
| Qualité des contrôles | Règles personnalisées | Validations structurées |
| Facilité d'utilisation | Déjà utilisé dans le projet | Apprentissage supplémentaire |
| Reproductibilité | Code Python réexécutable | Validations configurables |
| Maintenance | Règles maintenues dans le code | Règles organisées en validations |
| Temps de mise en œuvre | Faible pour ce projet | Configuration supplémentaire |
| Confidentialité | Traitement local possible | Traitement local possible selon configuration |

### Analyse comparative

Pandas et NumPy permettent de développer des contrôles
adaptés aux règles métier de Bottleneck.

Great Expectations propose une approche structurée
de la validation des données, utile notamment pour
des contrôles réguliers et documentés.

### Décision provisoire

La solution Pandas/NumPy est privilégiée pour
le notebook amélioré du Projet 6.

Elle répond aux besoins actuels et permet
de réutiliser le code déjà testé.

Great Expectations reste une solution envisageable
pour une évolution future du dispositif.

### Sources documentaires à consulter

- Pandas : https://pandas.pydata.org/docs/
- NumPy : https://numpy.org/doc/stable/
- Great Expectations : https://docs.greatexpectations.io/

### Limites de la comparaison

Cette première comparaison est qualitative.

Les temps d'exécution, les coûts de maintenance
et la facilité de déploiement n'ont pas encore
fait l'objet de mesures comparatives.



## Étape 1.12 : Synthèse de la veille et justification du choix

### 1. Besoin identifié

L'analyse des données de Bottleneck a mis en évidence
des incohérences entre les quantités disponibles
et les statuts de stock.

La méthode initiale ne détectait pas certaines
anomalies de qualité, notamment les stocks négatifs.

Le besoin métier consiste à fiabiliser les contrôles
pour améliorer la qualité des informations de stock.

### 2. Solutions étudiées

Deux solutions ont été comparées :

- Pandas et NumPy : contrôles personnalisés en Python.
- Great Expectations : validation structurée des données.

La comparaison a porté sur la qualité des contrôles,
la facilité d'utilisation, la reproductibilité,
la maintenance, le temps de mise en œuvre
et la confidentialité des données.

### 3. Résultats observés

La méthode Pandas/NumPy améliorée a été testée
sur les 825 produits du fichier ERP.

Elle a permis de détecter 4 anomalies,
contre 2 avec la méthode initiale.

Les anomalies supplémentaires correspondent
à deux quantités de stock négatives.

Great Expectations a été étudié comme solution
alternative, mais n'a pas été testé dans ce projet.

### 4. Décision technique

Pandas et NumPy sont retenus pour le contrôle
qualité amélioré des stocks de Bottleneck.

Ce choix repose sur :

- les résultats obtenus sur les données ERP ;
- la possibilité de personnaliser les règles métier ;
- la réutilisation du code Python existant ;
- la simplicité de mise en œuvre dans Jupyter ;
- la possibilité de reproduire les contrôles.

### 5. Limites de la décision

La comparaison entre les deux solutions
reste principalement qualitative.

Aucun benchmark de performance n'a été réalisé
entre Pandas/NumPy et Great Expectations.

Great Expectations pourra être évalué
dans une évolution future du projet.

### 6. Sources de veille

Documentation officielle Pandas :
https://pandas.pydata.org/docs/

Documentation officielle NumPy :
https://numpy.org/doc/stable/

Documentation officielle Great Expectations :
https://docs.greatexpectations.io/

Ces sources constituent la base documentaire
de la veille technologique.

La date de consultation et les informations
précises retenues seront consignées
dans la documentation finale.



# Étape 2 : Cahier des charges fonctionnel

## Étape 2.1 : Contexte et expression du besoin métier

### 1. Présentation du contexte

Bottleneck est une entreprise spécialisée dans la vente
de vins et spiritueux.

Dans le cadre du Projet 6, une analyse des données
issues de son ERP et de sa boutique en ligne a été
réalisée afin d'améliorer la fiabilité des informations
produits et de faciliter leur exploitation.

Le Projet 13 consiste à améliorer le notebook
d'analyse existant grâce à une utilisation critique
et documentée de l'intelligence artificielle.

### 2. Problématique métier

Comment fiabiliser les informations de stock
de Bottleneck afin d'identifier les incohérences
et de faciliter les décisions de gestion ?

### 3. Parties prenantes

**Responsable de la boutique :**
Utilise les résultats pour identifier les produits
nécessitant une vérification.

**Équipe de gestion des stocks :**
Analyse les anomalies et vérifie les informations
dans le système ERP.

**Data Analyst :**
Développe les contrôles, analyse les résultats,
documente les méthodes et formule des recommandations.

### 4. Besoins fonctionnels

Le livrable amélioré doit permettre de :

- Charger les données ERP.
- Vérifier la présence des informations nécessaires.
- Détecter les quantités de stock manquantes.
- Détecter les quantités négatives.
- Identifier les statuts manquants ou inconnus.
- Détecter les incohérences entre quantité et statut.
- Produire un tableau des anomalies.
- Calculer des indicateurs de qualité des données.
- Préserver les données sources originales.

### 5. Valeur métier attendue

L'amélioration doit permettre :

- d'identifier plus facilement les produits à vérifier ;
- de renforcer la fiabilité des contrôles ;
- de faciliter les décisions de gestion des stocks ;
- de réduire les risques d'erreurs d'interprétation ;
- de disposer d'une méthode documentée et réutilisable.

### 6. Résultats déjà observés

Sur 825 produits ERP analysés :

- Méthode initiale : 2 anomalies détectées.
- Méthode améliorée : 4 anomalies détectées.
- Anomalies supplémentaires : 2 stocks négatifs.

Ces résultats démontrent une meilleure couverture
des catégories d'anomalies étudiées, sans prouver
à eux seuls une amélioration de la qualité
des données sources.



## Étape 2.2 : Définition du périmètre et des contraintes

### 1. Périmètre fonctionnel

Le projet porte sur l'amélioration du contrôle qualité
des données de stock de Bottleneck.

Les fonctionnalités incluses sont :

- Importation du fichier ERP au format Excel.
- Vérification des colonnes nécessaires.
- Détection des quantités manquantes.
- Détection des quantités négatives.
- Identification des statuts manquants.
- Identification des statuts inconnus.
- Contrôle de cohérence entre quantité et statut.
- Calcul des indicateurs de qualité.
- Production d'un tableau des anomalies.
- Documentation des contrôles réalisés.

### 2. Éléments exclus du périmètre

Les actions suivantes ne sont pas prévues :

- Modification automatique des données ERP.
- Correction directe des stocks dans le système source.
- Développement d'une application web.
- Déploiement d'une solution en production.
- Remplacement du système ERP existant.

### 3. Contraintes techniques

**Environnement :**
Jupyter Notebook et Python.

**Bibliothèques :**
Pandas et NumPy.

**Sources de données :**
Fichiers Excel issus du Projet 6.

**Confidentialité :**
Les données doivent être traitées dans un environnement
approprié, sans transmission non autorisée.

**Intégrité :**
Les fichiers sources doivent rester inchangés.

**Reproductibilité :**
Les contrôles doivent pouvoir être réexécutés
sur des données de structure compatible.

### 4. Contraintes métier

Les règles de gestion retenues sont :

- Stock supérieur à 0 : instock.
- Stock égal à 0 : outofstock.
- Stock négatif : anomalie de quantité.
- Quantité manquante : anomalie de qualité.
- Statut manquant : anomalie de qualité.
- Statut inconnu : anomalie de qualité.

Les incohérences doivent être signalées sans
correction automatique.

### 5. Livrables attendus

- Notebook Python amélioré et documenté.
- Tableau des anomalies détectées.
- Indicateurs de qualité des données.
- Comparaison des méthodes de contrôle.
- Justification du choix technique.
- Cahier des charges fonctionnel.
- Documentation des expérimentations avec l'IA.

### 6. Hypothèses et limites

Le contrôle repose sur les règles métier définies
pour les données étudiées.

Les résultats ne garantissent pas l'absence
d'autres anomalies dans le système ERP.

Les anomalies détectées devront être vérifiées
par les responsables métier avant toute correction.



## Étape 2.3 : Exigences fonctionnelles et critères d'acceptation

### 1. Objectif

Définir les fonctionnalités attendues du notebook amélioré et les conditions permettant de vérifier leur bon fonctionnement.

### 2. Exigences fonctionnelles

| ID | Exigence fonctionnelle | Priorité |
|---|---|---|
| EF01 | Charger les données ERP depuis un fichier Excel | Haute |
| EF02 | Vérifier la présence des colonnes nécessaires | Haute |
| EF03 | Détecter les quantités de stock manquantes | Haute |
| EF04 | Détecter les quantités de stock négatives | Haute |
| EF05 | Identifier les statuts de stock manquants | Haute |
| EF06 | Identifier les statuts de stock inconnus | Haute |
| EF07 | Détecter les incohérences entre quantité et statut | Haute |
| EF08 | Calculer le nombre et le taux d'anomalies | Haute |
| EF09 | Produire un tableau détaillé des anomalies | Haute |
| EF10 | Préserver les données ERP originales | Haute |
| EF11 | Documenter les règles et les résultats | Moyenne |

### 3. Critères d'acceptation

**CA01 - Chargement des données**

Le notebook doit pouvoir importer un fichier Excel ERP valide et afficher le nombre de produits chargés.

**CA02 - Vérification des colonnes**

Le programme doit vérifier la présence des colonnes `product_id`, `stock_quantity` et `stock_status`.

**CA03 - Quantités manquantes**

Toute quantité absente doit être identifiée comme une anomalie de qualité.

**CA04 - Quantités négatives**

Toute quantité inférieure à zéro doit être signalée comme une anomalie de quantité.

**CA05 - Statuts manquants**

Tout statut absent doit être détecté et signalé.

**CA06 - Statuts inconnus**

Tout statut différent de `instock` et `outofstock` doit être signalé.

**CA07 - Cohérence quantité/statut**

Le contrôle doit identifier les cas suivants :

- Quantité positive avec statut `outofstock`.
- Quantité nulle avec statut `instock`.

**CA08 - Indicateurs de qualité**

Le notebook doit afficher :

- Le nombre total de produits analysés.
- Le nombre de produits présentant une anomalie.
- Le taux de produits présentant une anomalie.
- Le nombre d'anomalies par catégorie.

**CA09 - Tableau des anomalies**

Le tableau doit permettre d'identifier les produits concernés, leurs quantités, leurs statuts et les types d'anomalies détectées.

**CA10 - Intégrité des données**

Les fichiers ERP originaux doivent rester inchangés après l'exécution des contrôles.

**CA11 - Documentation**

Les règles métier, les traitements et les résultats doivent être expliqués dans le notebook.

### 4. Résultats de référence

Les premiers contrôles réalisés sur les données ERP de Bottleneck ont donné les résultats suivants :

| Indicateur | Méthode initiale | Méthode améliorée |
|---|---:|---:|
| Produits analysés | 825 | 825 |
| Produits avec anomalie | 2 | 4 |
| Taux d'anomalies | 0,24 % | 0,48 % |
| Stocks négatifs détectés | 0 | 2 |

Ces résultats constituent une référence pour les tests d'acceptation sur le fichier ERP étudié.

### 5. Conditions de validation

Une exigence sera considérée comme validée lorsque :

- Le code correspondant aura été exécuté.
- Le résultat obtenu aura été vérifié.
- Les anomalies attendues auront été identifiées.
- Les données originales auront été préservées.

Les tests sur des données fictives permettront également de vérifier les cas particuliers.

### 6. Limites

Les critères d'acceptation reposent sur les règles métier retenues pour ce projet.

La détection d'une anomalie ne signifie pas qu'une correction automatique doit être effectuée.

Toute correction des données sources nécessite une vérification métier préalable.



# Étape 2.4 : Plan de tests et validation fonctionnelle

## 1. Objectif

Vérifier que le notebook amélioré répond aux exigences fonctionnelles définies dans le cahier des charges de Bottleneck.

Les tests permettent de contrôler la fiabilité des traitements, la détection des anomalies et la conformité des résultats aux règles métier.

## 2. Méthodologie

Les tests sont organisés à partir des exigences EF01 à EF11 et des critères d'acceptation CA01 à CA11.

Deux catégories de données sont utilisées :

- Les données ERP réelles de Bottleneck, comprenant 825 produits.
- Des données fictives permettant de tester des situations particulières.

Les données sources originales ne doivent pas être modifiées.

## 3. Plan de tests fonctionnels

| Test | Exigence | Vérification attendue |
|---|---|---|
| T01 | EF01 | Charger correctement le fichier ERP Excel |
| T02 | EF02 | Vérifier les colonnes obligatoires |
| T03 | EF03 | Détecter les quantités manquantes |
| T04 | EF04 | Détecter les quantités négatives |
| T05 | EF05 | Détecter les statuts manquants |
| T06 | EF06 | Détecter les statuts inconnus |
| T07 | EF07 | Détecter les incohérences quantité/statut |
| T08 | EF08 | Calculer les indicateurs de qualité |
| T09 | EF09 | Afficher le tableau détaillé des anomalies |
| T10 | EF10 | Préserver les données ERP originales |
| T11 | EF11 | Documenter les règles et les résultats |

## 4. Résultats de référence

Les contrôles précédents sur les données ERP ont produit les résultats suivants :

- Produits analysés : 825.
- Anomalies détectées par la méthode initiale : 2.
- Anomalies détectées par la méthode améliorée : 4.
- Incohérences de statut : 2.
- Quantités négatives : 2.
- Taux de produits présentant une anomalie : 0,48 %.

Ces résultats serviront de référence pour les tests de validation.

## 5. Conditions de réussite

Un test sera considéré comme validé lorsque :

- Le code Python s'exécute correctement.
- Le résultat observé correspond au résultat attendu.
- Les anomalies prévues sont correctement identifiées.
- Les données ERP originales sont préservées.

Les tests non exécutés ou non concluants seront signalés.

## 6. Traçabilité

Chaque test devra être associé à une exigence fonctionnelle, à un résultat attendu, à un résultat observé et à un statut de validation.

Cette démarche permettra de justifier la conformité du notebook amélioré au cahier des charges.


In [ ]:

# ============================================================
# ETAPE 2.4 - CELLULE 1
# TESTS T01 ET T02 : DONNEES ERP ET COLONNES OBLIGATOIRES
# ============================================================

import pandas as pd

print("=== VALIDATION FONCTIONNELLE : T01 ET T02 ===")

# Colonnes indispensables au contrôle des stocks
colonnes_obligatoires = [
    "product_id",
    "stock_quantity",
    "stock_status"
]

# Recherche du DataFrame ERP déjà chargé dans le notebook
noms_possibles = [
    "df_erp",
    "erp",
    "df_stock",
    "df"
]

df_test = None
nom_dataframe = None

for nom in noms_possibles:
    objet = globals().get(nom)

    if isinstance(objet, pd.DataFrame):
        if all(
            col in objet.columns
            for col in colonnes_obligatoires
        ):
            df_test = objet.copy()
            nom_dataframe = nom
            break

# T01 : Vérification du chargement
if df_test is not None:
    print("\nT01 - Chargement des données : VALIDE")
    print("DataFrame utilisé :", nom_dataframe)
    print("Nombre de produits :", len(df_test))
else:
    print("\nT01 - Chargement : NON VERIFIE")
    print("Aucun DataFrame ERP compatible trouvé.")

# T02 : Vérification des colonnes
if df_test is not None:

    colonnes_manquantes = [
        col for col in colonnes_obligatoires
        if col not in df_test.columns
    ]

    if not colonnes_manquantes:
        print("\nT02 - Colonnes obligatoires : VALIDE")
        print("Colonnes vérifiées :", colonnes_obligatoires)
    else:
        print("\nT02 - Colonnes obligatoires : ECHEC")
        print("Colonnes manquantes :", colonnes_manquantes)

print("\n=== FIN DES TESTS T01 ET T02 ===")


In [ ]:

# ============================================================
# ETAPE 2.4 - CELLULE 2
# TESTS T03 ET T04 : CONTROLE DES QUANTITES
# ============================================================

import pandas as pd
from IPython.display import display

print("=== VALIDATION FONCTIONNELLE : T03 ET T04 ===")

# Copie de travail pour préserver les données originales
df_controle = df_test.copy()

# Conversion des quantités en valeurs numériques
quantites = pd.to_numeric(
    df_controle["stock_quantity"],
    errors="coerce"
)

# T03 : Quantités manquantes ou non numériques
df_controle["quantite_manquante"] = quantites.isna()

nb_manquantes = df_controle["quantite_manquante"].sum()

print("\nT03 - Quantités manquantes")
print("Nombre détecté :", nb_manquantes)

# T04 : Quantités négatives
df_controle["quantite_negative"] = quantites < 0

nb_negatives = df_controle["quantite_negative"].sum()

print("\nT04 - Quantités négatives")
print("Nombre détecté :", nb_negatives)

# Vérification des résultats de référence
print("\n=== COMPARAISON AUX RESULTATS ATTENDUS ===")

print(
    "T03 :",
    "VALIDE" if nb_manquantes == 0 else "A VERIFIER"
)

print(
    "T04 :",
    "VALIDE" if nb_negatives == 2 else "A VERIFIER"
)

# Affichage des produits présentant une anomalie
colonnes_affichage = [
    "product_id",
    "stock_quantity",
    "stock_status",
    "quantite_manquante",
    "quantite_negative"
]

anomalies_quantites = df_controle[
    df_controle["quantite_manquante"]
    | df_controle["quantite_negative"]
]

print("\n=== PRODUITS A VERIFIER ===")

display(anomalies_quantites[colonnes_affichage])

print("\n=== FIN DES TESTS T03 ET T04 ===")


In [ ]:

# ============================================================
# ETAPE 2.4 - CELLULE 3
# TESTS T05 ET T06 : CONTROLE DES STATUTS DE STOCK
# ============================================================

import pandas as pd
from IPython.display import display

print("=== VALIDATION FONCTIONNELLE : T05 ET T06 ===")

# Copie de travail pour préserver les données ERP
df_statuts = df_test.copy()

# Normalisation des statuts pour le contrôle
statuts = (
    df_statuts["stock_status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Statuts autorisés
statuts_valides = ["instock", "outofstock"]

# T05 : Détection des statuts manquants
df_statuts["statut_manquant"] = (
    statuts.isna() | statuts.eq("").fillna(False)
)

nb_statuts_manquants = int(
    df_statuts["statut_manquant"].sum()
)

print("\nT05 - Statuts manquants")
print("Nombre détecté :", nb_statuts_manquants)

# T06 : Détection des statuts inconnus
df_statuts["statut_inconnu"] = (
    ~statuts.isin(statuts_valides)
    & ~df_statuts["statut_manquant"]
)

nb_statuts_inconnus = int(
    df_statuts["statut_inconnu"].sum()
)

print("\nT06 - Statuts inconnus")
print("Nombre détecté :", nb_statuts_inconnus)

# Comparaison aux résultats de référence
print("\n=== COMPARAISON AUX RESULTATS ATTENDUS ===")

print(
    "T05 :",
    "VALIDE" if nb_statuts_manquants == 0
    else "A VERIFIER"
)

print(
    "T06 :",
    "VALIDE" if nb_statuts_inconnus == 0
    else "A VERIFIER"
)

# Affichage des anomalies éventuelles
anomalies_statuts = df_statuts[
    df_statuts["statut_manquant"]
    | df_statuts["statut_inconnu"]
]

print("\n=== PRODUITS A VERIFIER ===")

colonnes_affichage = [
    "product_id",
    "stock_quantity",
    "stock_status",
    "statut_manquant",
    "statut_inconnu"
]

display(anomalies_statuts[colonnes_affichage])

print("\n=== FIN DES TESTS T05 ET T06 ===")


In [ ]:

# ============================================================
# ETAPE 2.4 - CELLULE 4
# TEST T07 : COHERENCE ENTRE QUANTITE ET STATUT
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

print("=== VALIDATION FONCTIONNELLE : T07 ===")

# Copie des données ERP originales
df_t07 = df_test.copy()

# Conversion des quantités en valeurs numériques
quantites = pd.to_numeric(
    df_t07["stock_quantity"],
    errors="coerce"
)

# Normalisation des statuts
statuts = (
    df_t07["stock_status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Calcul du statut attendu
# Les quantités négatives ou manquantes restent exclues
# de la comparaison des statuts.
df_t07["statut_attendu"] = np.select(
    [
        quantites > 0,
        quantites == 0
    ],
    [
        "instock",
        "outofstock"
    ],
    default="non_applicable"
)

# Détection des incohérences
statut_valide = statuts.isin(["instock", "outofstock"])
quantite_valide = quantites.notna() & (quantites >= 0)

df_t07["incoherence_statut"] = (
    quantite_valide
    & statut_valide
    & (
        statuts != df_t07["statut_attendu"]
    ).fillna(False)
)

# Nombre d'incohérences
nb_incoherences = int(
    df_t07["incoherence_statut"].sum()
)

print("\nT07 - Incohérences quantité/statut")
print("Nombre détecté :", nb_incoherences)

# Vérification du résultat de référence
print("\n=== COMPARAISON AU RESULTAT ATTENDU ===")

print(
    "T07 :",
    "VALIDE" if nb_incoherences == 2
    else "A VERIFIER"
)

# Affichage des produits concernés
anomalies_t07 = df_t07[
    df_t07["incoherence_statut"]
]

print("\n=== PRODUITS PRESENTANT UNE INCOHERENCE ===")

display(
    anomalies_t07[
        [
            "product_id",
            "stock_quantity",
            "stock_status",
            "statut_attendu",
            "incoherence_statut"
        ]
    ]
)

print("\n=== FIN DU TEST T07 ===")


In [ ]:

# ============================================================
# DIAGNOSTIC DU TEST T07
# Verification des donnees utilisees
# ============================================================

print("=== DIAGNOSTIC T07 ===")

# Comparaison des deux DataFrames
print("\nNombre de lignes dans df_erp :", len(df_erp))
print("Nombre de lignes dans df_test :", len(df_test))

# Verification des produits attendus
produits = [4039, 4885]

for nom, df in [("df_erp", df_erp), ("df_test", df_test)]:

    print(f"\n=== PRODUITS DANS {nom} ===")

    ids = pd.to_numeric(df["product_id"], errors="coerce")

    display(
        df.loc[
            ids.isin(produits),
            ["product_id", "stock_quantity", "stock_status"]
        ]
    )

# Verification des statuts
print("\n=== STATUTS DANS df_test ===")

print(df_test["stock_status"].value_counts(dropna=False))

print("\n=== FIN DU DIAGNOSTIC ===")


In [ ]:

# ============================================================
# ETAPE 2.4 - VERIFICATION DES STATUTS ERP
# ============================================================

print("=== VERIFICATION DES STATUTS ===")

# Produits initialement identifies comme incoherents
produits = [4039, 4885]

# Afficher les valeurs actuelles
display(
    df_erp.loc[
        df_erp["product_id"].isin(produits),
        ["product_id", "stock_quantity", "stock_status"]
    ]
)

# Verification de la coherence sur les 825 produits
quantites = pd.to_numeric(
    df_erp["stock_quantity"],
    errors="coerce"
)

statuts = df_erp["stock_status"]

incoherences = (
    ((quantites > 0) & (statuts == "outofstock"))
    |
    ((quantites == 0) & (statuts == "instock"))
)

print("\nNombre d'incoherences actuelles :", incoherences.sum())

print("\n=== FIN DE LA VERIFICATION ===")


In [ ]:

# ============================================================
# ETAPE 2.4 - T07 : VERIFICATION DU FICHIER ERP SOURCE
# ============================================================

from pathlib import Path
import pandas as pd

# Rechercher les fichiers Excel disponibles
fichiers = list(Path(".").glob("*.xlsx"))

print("=== FICHIERS EXCEL DISPONIBLES ===")

for i, fichier in enumerate(fichiers, start=1):
    print(f"{i}. {fichier.name}")

print("\n=== FIN DE LA RECHERCHE ===")


In [ ]:

# ============================================================
# ETAPE 2.4 - T07 : CONTROLE DU FICHIER ERP ORIGINAL
# ============================================================

import pandas as pd

# 1. Charger le fichier ERP sans modifier les données
df_erp_original = pd.read_excel("erp.xlsx")

print("=== VERIFICATION DU FICHIER ERP ORIGINAL ===")

print("Nombre de lignes :", len(df_erp_original))

# 2. Vérifier les colonnes disponibles
print("\nColonnes disponibles :")
print(df_erp_original.columns.tolist())

# 3. Rechercher les deux produits concernés
produits = [4039, 4885]

colonnes = [
    "product_id",
    "stock_quantity",
    "stock_status"
]

print("\n=== PRODUITS DANS LE FICHIER ORIGINAL ===")

display(
    df_erp_original.loc[
        df_erp_original["product_id"].isin(produits),
        colonnes
    ]
)

print("\n=== FIN DE LA VERIFICATION ===")


In [ ]:

# ============================================================
# ETAPE 2.4 - T07 : VALIDATION SUR LE FICHIER ERP ORIGINAL
# ============================================================

import pandas as pd
import numpy as np

# 1. Charger une copie indépendante du fichier original
df_t07 = pd.read_excel("erp.xlsx").copy()

# 2. Convertir les quantités pour le contrôle
quantites = pd.to_numeric(
    df_t07["stock_quantity"],
    errors="coerce"
)

# 3. Calculer le statut attendu uniquement pour
# les quantités valides et positives ou nulles
df_t07["statut_attendu"] = np.select(
    [
        quantites > 0,
        quantites == 0
    ],
    [
        "instock",
        "outofstock"
    ],
    default=None
)

# 4. Identifier les incohérences quantité/statut
df_t07["incoherence_statut"] = (
    quantites.ge(0)
    & quantites.notna()
    & df_t07["stock_status"].isin(
        ["instock", "outofstock"]
    )
    & df_t07["stock_status"].ne(
        df_t07["statut_attendu"]
    )
)

# 5. Afficher les résultats
nb_incoherences = int(
    df_t07["incoherence_statut"].sum()
)

print("=== VALIDATION FONCTIONNELLE : T07 ===")

print("Produits analysés :", len(df_t07))
print("Incohérences détectées :", nb_incoherences)

print("\n=== COMPARAISON AU RESULTAT ATTENDU ===")

if nb_incoherences == 2:
    print("T07 : VALIDE")
else:
    print("T07 : A VERIFIER")

print("\n=== PRODUITS PRESENTANT UNE INCOHERENCE ===")

display(
    df_t07.loc[
        df_t07["incoherence_statut"],
        [
            "product_id",
            "stock_quantity",
            "stock_status",
            "statut_attendu",
            "incoherence_statut"
        ]
    ]
)

print("\n=== FIN DU TEST T07 ===")


In [ ]:

# ============================================================
# ETAPE 2.4 - T08 : VALIDATION DES INDICATEURS DE QUALITE
# ============================================================

import pandas as pd

# 1. Charger une copie du fichier ERP original
df_t08 = pd.read_excel("erp.xlsx").copy()

# 2. Préparer les colonnes de contrôle
q = pd.to_numeric(
    df_t08["stock_quantity"], errors="coerce"
)

s = df_t08["stock_status"]

# 3. Identifier les catégories d'anomalies
df_t08["quantite_manquante"] = q.isna()
df_t08["quantite_negative"] = q.lt(0)
df_t08["statut_manquant"] = s.isna()

df_t08["statut_inconnu"] = (
    s.notna()
    & ~s.isin(["instock", "outofstock"])
)

df_t08["incoherence_statut"] = (
    ((q > 0) & (s == "outofstock"))
    | ((q == 0) & (s == "instock"))
)

# 4. Calculer les indicateurs
colonnes_anomalies = [
    "quantite_manquante",
    "quantite_negative",
    "statut_manquant",
    "statut_inconnu",
    "incoherence_statut"
]

df_t08["anomalie_globale"] = (
    df_t08[colonnes_anomalies].any(axis=1)
)

total = len(df_t08)
nb_anomalies = int(df_t08["anomalie_globale"].sum())
taux = (nb_anomalies / total * 100) if total else 0

print("=== VALIDATION FONCTIONNELLE : T08 ===")

print("Produits analysés :", total)
print("Produits avec anomalie :", nb_anomalies)
print(f"Taux d'anomalies : {taux:.2f} %")

print("\n=== DETAIL PAR CATEGORIE ===")
print(df_t08[colonnes_anomalies].sum())

# 5. Comparer aux résultats de référence
validation = (
    total == 825
    and nb_anomalies == 4
    and int(df_t08["quantite_negative"].sum()) == 2
    and int(df_t08["incoherence_statut"].sum()) == 2
)

print("\n=== COMPARAISON AUX RESULTATS ATTENDUS ===")

if validation:
    print("T08 : VALIDE")
else:
    print("T08 : A VERIFIER")

print("\n=== FIN DU TEST T08 ===")


In [ ]:

# ============================================================
# ETAPE 2.4 - T09 : TABLEAU DETAILLE DES ANOMALIES
# ============================================================

# 1. Reutiliser les resultats valides du test T08
df_t09 = df_t08.copy()

# 2. Selectionner les produits avec anomalie
df_anomalies_t09 = df_t09.loc[
    df_t09["anomalie_globale"]
].copy()

# 3. Colonnes du tableau
colonnes_t09 = [
    "product_id",
    "stock_quantity",
    "stock_status",
    "quantite_manquante",
    "quantite_negative",
    "statut_manquant",
    "statut_inconnu",
    "incoherence_statut"
]

print("=== VALIDATION FONCTIONNELLE : T09 ===")

print("Produits analyses :", len(df_t09))
print("Produits avec anomalie :", len(df_anomalies_t09))

print("\n=== TABLEAU DETAILLE DES ANOMALIES ===")

display(df_anomalies_t09[colonnes_t09])

# 4. Verification des produits attendus
produits_attendus = {4039, 4885, 4973, 5700}

produits_detectes = set(
    df_anomalies_t09["product_id"].tolist()
)

validation_t09 = (
    len(df_anomalies_t09) == 4
    and produits_detectes == produits_attendus
)

print("\n=== COMPARAISON AUX RESULTATS ATTENDUS ===")

if validation_t09:
    print("T09 : VALIDE")
else:
    print("T09 : A VERIFIER")

print("\n=== FIN DU TEST T09 ===")


In [ ]:

# ============================================================
# ETAPE 2.4 - T10 : INTEGRITE DES DONNEES ERP
# ============================================================

import pandas as pd

print("=== VALIDATION FONCTIONNELLE : T10 ===")

# 1. Charger le fichier ERP original
df_source_t10 = pd.read_excel("erp.xlsx")

# 2. Creer une copie independante
df_copie_t10 = df_source_t10.copy(deep=True)

# 3. Executer un controle sur la copie
df_copie_t10["controle_stock"] = (
    pd.to_numeric(
        df_copie_t10["stock_quantity"],
        errors="coerce"
    ) < 0
)

# 4. Verifier que la source en memoire reste intacte
source_intacte = (
    "controle_stock" not in df_source_t10.columns
)

# 5. Relire le fichier original
df_verification_t10 = pd.read_excel("erp.xlsx")

fichier_coherent = df_source_t10.equals(
    df_verification_t10
)

print("Produits analyses :", len(df_source_t10))

print("\n=== VERIFICATION DE L'INTEGRITE ===")

print("Source en memoire preservee :", source_intacte)
print("Fichier relu identique :", fichier_coherent)

print("\n=== COMPARAISON AU RESULTAT ATTENDU ===")

if source_intacte and fichier_coherent:
    print("T10 : VALIDE")
else:
    print("T10 : A VERIFIER")

print("\n=== FIN DU TEST T10 ===")


In [ ]:

# ============================================================
# ETAPE 2.4 - T11 : DOCUMENTATION ET TRACABILITE
# ============================================================

import pandas as pd
from IPython.display import display

print("=== VALIDATION FONCTIONNELLE : T11 ===")

# 1. Documentation des regles metier
regles_metier = pd.DataFrame({
    "Regle": [
        "R01", "R02", "R03", "R04", "R05", "R06"
    ],
    "Description": [
        "Quantite > 0 : instock",
        "Quantite = 0 : outofstock",
        "Quantite negative : anomalie",
        "Quantite manquante : anomalie",
        "Statut manquant ou inconnu : anomalie",
        "Incoherence quantite/statut : anomalie"
    ]
})

print("\n=== REGLES METIER DOCUMENTEES ===")
display(regles_metier)

# 2. Resultats de reference
resultats = pd.DataFrame({
    "Indicateur": [
        "Produits analyses",
        "Produits avec anomalie",
        "Taux d'anomalies",
        "Quantites negatives",
        "Incoherences de statut"
    ],
    "Valeur": [
        825,
        4,
        "0.48 %",
        2,
        2
    ]
})

print("\n=== RESULTATS DOCUMENTES ===")
display(resultats)

# 3. Verification de la documentation creee
documentation_presente = (
    len(regles_metier) == 6
    and len(resultats) == 5
    and regles_metier.notna().all().all()
    and resultats.notna().all().all()
)

print("\n=== CONTROLE DE DOCUMENTATION ===")

if documentation_presente:
    print("Documentation de synthese : PRESENTE")
    print("T11 : DOCUMENTATION DE SYNTHESE CREEE")
else:
    print("T11 : A VERIFIER")

print("\n=== FIN DU TEST T11 ===")
